## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [41]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")
#groq_api_key



In [42]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000139B97B3CE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000139BC9362C0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [43]:
from langchain_core.messages import HumanMessage

response = model.invoke([
    HumanMessage(content="Hi, my name is Tushar and I am an AI Engineer")
])

print(response.content)


Hello Tushar! 👋 Great to meet an AI engineer. How can I assist you today? Whether it’s brainstorming ideas, troubleshooting code, or exploring new AI concepts, I’m here to help!


In [44]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Tushar and I am a AI Engineer"),
        AIMessage(content="Hello Tushar! It's nice to meet you. \n\nAs a AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='You’re Tushar, and you’re an AI Engineer.', additional_kwargs={'reasoning_content': 'User: "Hey What\'s my name and what do I do?" They previously introduced themselves: "Hi , My name is Tushar and I am a AI Engineer". So answer: name Tushar, job AI Engineer. Should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 152, 'total_tokens': 225, 'completion_time': 0.080874484, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.008388705, 'prompt_tokens_details': None, 'queue_time': 0.3082864, 'total_time': 0.089263189}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0dde5-0ee8-7eb0-b66b-94110a323e91-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 73, 'total_tokens': 225, 'output_token_details': {'reasoning': 51}})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [45]:
!pip install langchain_community

In [46]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

c:\Users\Tushar Nile\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [47]:
config={"configurable":{"session_id":"chat1"}}

In [48]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Tushar and I am a  AI Engineer")],
    config=config
)

In [49]:
response.content

'Hello Tushar! 👋  \nNice to meet an AI engineer—what exciting projects are you working on right now? Whether you need help brainstorming ideas, debugging code, exploring new models, or just want to chat about the latest AI trends, I’m here to help. Let me know what’s on your mind!'

In [50]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='You’re Tushar!', additional_kwargs={'reasoning_content': 'User asks: "What\'s my name?" We know from earlier that user said "Hi , My name is Tushar and I am a AI Engineer". So answer: "Your name is Tushar." We can respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 165, 'total_tokens': 228, 'completion_time': 0.064624797, 'completion_tokens_details': {'reasoning_tokens': 48}, 'prompt_time': 0.007942239, 'prompt_tokens_details': None, 'queue_time': 0.310535897, 'total_time': 0.072567036}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c9afb2bdb4', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0dde5-17ec-75f1-9567-dd6555b1c5b4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_tokens': 63, 'total_tokens': 228, 'output_token_details': {'reasoning': 48}})

In [51]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'I’m not sure—what’s your name?'

In [52]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

'Nice to meet you, John! How can I help you today?'

In [53]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [54]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [55]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Tushar")]})

AIMessage(content='Hello Tushar! 👋 Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond as a helpful assistant, greet the user, possibly ask for what they need. The user just introduced themselves. We should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 99, 'total_tokens': 159, 'completion_time': 0.068669044, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.032272108, 'prompt_tokens_details': None, 'queue_time': 0.346651146, 'total_time': 0.100941152}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1c0f5282a2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0dde5-1f95-7df2-8a29-0bbcd80076f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 60, 'total_tokens': 159, 'output_token_details': {'reasoning': 32}})

In [56]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Users\Tushar Nile\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [57]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Tushar")],
    config=config
)

response

AIMessage(content='Hello Tushar! 👋 Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond to greeting. The user says "Hi My name is Tushar". We can reply: "Hi Tushar, nice to meet you. How can I help?"'}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 99, 'total_tokens': 167, 'completion_time': 0.075754549, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.004707171, 'prompt_tokens_details': None, 'queue_time': 0.310486594, 'total_time': 0.08046172}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3cdd24b83b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0dde5-21b4-79d2-b9dc-a76328153e58-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 68, 'total_tokens': 167, 'output_token_details': {'reasoning': 40}})

In [58]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'You’re Tushar!'

In [59]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [60]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Tushar")],"language":"Hindi"})
response.content

'नमस्ते तुषार जी! आपका स्वागत है। मैं आपकी किस प्रकार सहायता कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [61]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

c:\Users\Tushar Nile\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [62]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते कृष्ण जी! आप कैसे हैं? आप मुझसे किसी विषय पर बात करना चाहेंगे या किसी प्रश्न का उत्तर चाहते हैं? मैं आपकी सहायता के लिए यहाँ हूँ।'

In [63]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [64]:
response.content

'आपका नाम कृष्ण है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [65]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [66]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I’m not sure—what flavors do you usually enjoy? If you give me a few hints (like sweet, salty, fruity, chocolatey, etc.), I can suggest some ice cream flavors you might like!'

In [67]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked for the sum of 2 and 2.'

In [68]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

c:\Users\Tushar Nile\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [69]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'I don’t actually know your name—if you’d like to tell me, I’ll remember it for the rest of our chat!'

In [70]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'I don’t see a math problem in our conversation so far. Could you let me know which problem you’re referring to? Once I have the details, I’ll be happy to help!'